# ReBRAC Broad Validation — Colab Driver

> 文档对应：[docs/superpowers/specs/2026-05-04-rebrac-broad-validation-design.md](../docs/superpowers/specs/2026-05-04-rebrac-broad-validation-design.md) / [docs/superpowers/plans/2026-05-04-rebrac-broad-validation-plan.md](../docs/superpowers/plans/2026-05-04-rebrac-broad-validation-plan.md)
> 目的：以 anchor (Stage C finalist `crosscomp/s0/cross_stream/Re150/U=1.0/target=1.5/(β1=4, β2=2)` = 0.902 ± 0.021) 为锚，沿三轴 (data quality / sensor / task geometry) 做 8-spoke OFAT 广验。
> 收口要求：跑完所有 cell 后，按 §6 写入清单更新 plan / report / paper subsection。

## 执行摘要

| # | 任务 | 类型 | 预算 | 目的 |
|---|---|---|---|---|
| **S1** | 采集 7 个新数据集 (A1/A2/A3/B1/B2/C1/C3) | 采集 | ~6h L4 | 为 8 spokes 提供数据底座（A2 含 mix5050 episode-level concat） |
| **S2** | P1 训练 — 8 spokes × 2 seeds (=16 runs，含 A2 ReBRAC + A2-td3bc 双轨) | 训练 | ~8h L4 | 触发判定（spec §6.1）：决定哪些 spoke 进 P2 |
| **S3** | P2 conditional deepening — β refit (1-seed) + 5-seed 扩展 | 训练 | ~5–8h L4 | 仅对 P1 触发的 spoke 做深探（spec §6.2） |
| **S4** | 写 broad-validation 报告 + paper subsection | 写作 | ~2h | `docs/rebrac_broad_validation_report.md` (NEW) + `docs/rebrac_mainline_review.md §3.5` cross-link |

S1+S2 加起来 ~14h L4 必跑；S3 视触发情况 5–8h；S4 在本地。预算 19–22h L4 / 3–4 Colab Pro 会话。

## Trigger gate (spec §6.1)

每个 spoke 的 P1 (2-seed mean / std vs anchor 0.902 / 0.021) 触发以下任一条件即进 P2：
1. `|Δmean| > 5pp` (bidirectional)
2. `std > 2 × anchor_std = 4.2pp` (std blow-up)
3. **A2 only**: `|ReBRAC_A2 − TD3+BC_A2| < 5pp` (gap collapse)

## 与现有实验输出树的隔离

| 任务 | 输出根（与 Stage A–F 全部隔离） |
|---|---|
| S1 | `offline_data/{goalseek,mix5050,privileged,crosscomp_*}_*_re150_*_ep1000/`（per-spoke dataset name） |
| S2/S3 | `checkpoints/offline/rebrac/broad_validation/<spoke>/<pair>/seed_*`、`results/offline/rebrac/broad_validation/<spoke>/<pair>/{validation,selection,test}/` |
| 汇总 | `results/offline/rebrac/broad_validation/summaries/{p1_overview,p2_overview,trigger_decisions}.{csv,json}` |

每个 cell 是 `[skip]`-resume safe via `transitions.npz` / `trainer_state.json` / `agent_final.pt` / `selected_checkpoint.json` / per-seed test JSONs 存在性检查。


## 0. 环境 sanity check

In [ ]:
!lscpu | head -10
print()
!nvidia-smi


Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  12
On-line CPU(s) list:                     0-11
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   85

Wed May  6 07:37:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-U

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")


PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 1. 通用配置

S1–S3 共用 base env vars；各任务再自己 override。

In [2]:
import os

os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"
os.environ["HISTORY_LENGTH"]     = "4"
os.environ["OBJECTIVE"]          = "efficiency_v2"


## 2. 任务 S1 — 7 datasets 采集

### 2.1 scope / judgement

7 个新数据集（A1/A2/A3/B1/B2/C1/C3；anchor `crosscomp_s0_..._u10cross` 已有，0 算力）：

| spoke | dataset_name | collector | probe | geometry | flow | episodes | seed |
|---|---|---|---|---|---|---:|---:|
| A1 | `goalseek_s0_..._u10cross` | goalseek | s0 | cross_stream | Re150 single | 1000 | 0 |
| A2 | `mix5050_s0_..._u10cross` | goalseek + crosscomp 各 500 ep（episode-level concat）| s0 | cross_stream | Re150 single | 500+500 | 0, 1 |
| A3 | `privileged_s0_..._u10cross` | privileged | s0 | cross_stream | Re150 single | 1000 | 0 |
| B1 | `crosscomp_s1_..._u10cross` | crosscomp | s1 | cross_stream | Re150 single | 1000 | 0 |
| B2 | `crosscomp_s2_..._u10cross` | crosscomp | s2 | cross_stream | Re150 single | 1000 | 0 |
| C1 | `crosscomp_s0_..._u10upstream` | crosscomp | s0 | upstream | Re150 single | 1000 | 0 |
| C3 | `crosscomp_s0_..._re150tandem_u10cross` | crosscomp | s0 | cross_stream | Re150 tandem | 1000 | 0 |

预算：~6h L4（B2 16-D obs 略慢；A2 = 2 × 500ep 串行；其余 ~50 min/ep1000）。

判定（每个 dataset 必须满足）：

| 条件 | 阈值 | 说明 |
|---|---|---|
| `transitions.npz` + `metadata.json` + `sanity_card.json` 三件套齐全 | 必需 | 缺任一 → S1 失败 |
| `obs_dim` 与 `probe_layout` 匹配 | s0=10, s1=12, s2=16 | `sanity_card.obs_dim_matches_probe_layout` 必须 True |
| A2 metadata 含 `mix_components` (2 项) + `mix_strategy="episode_level"` + `task_sampler="anchor_distribution"` | spec §4.3 | 不可改用 `--policy-mixture`（后者 binomial 分配且缺 schema） |


### 2.2 单一 policy 6 spokes 采集

A1/A3/B1/B2/C1/C3 每个直接调 `scripts.collect_offline_data` 一次。Driver 跳过已有数据集（按 `transitions.npz` 存在性）。

In [3]:
from pathlib import Path
from scripts.broad_validation_spoke_registry import REGISTRY

SINGLE_POLICY_SPOKES = ["A1", "A3", "B1", "B2", "C1", "C3"]

In [ ]:
from pathlib import Path
from scripts.broad_validation_spoke_registry import REGISTRY

SINGLE_POLICY_SPOKES = ["A1", "A3", "B1", "B2", "C1", "C3"]
for spoke_id in SINGLE_POLICY_SPOKES:
    cfg = REGISTRY[spoke_id]
    out_dir = Path("offline_data") / cfg.dataset_name
    if (out_dir / "transitions.npz").exists():
        print(f"[skip] {spoke_id}: dataset exists at {out_dir}")
        continue
    print(f"\n[collect] {spoke_id} → {out_dir}")
    !python -m scripts.collect_offline_data \
        --policy {cfg.collector_policy} \
        --flow {cfg.flow_path} \
        --probe-layout {cfg.probe_layout} \
        --task-geometry {cfg.task_geometry} \
        --target-speed {cfg.target_speed} \
        --history-length 4 \
        --objective efficiency_v2 \
        --episodes 1000 \
        --seed 0 \
        --num-workers 8 \
        --output-dir {out_dir}



[collect] A1 → offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000
[collect] ep=125/1000 transitions=22224 success_rate=77.60% elapsed=341.2s
[collect] ep=250/1000 transitions=46061 success_rate=82.00% elapsed=348.8s
[collect] ep=375/1000 transitions=67723 success_rate=80.00% elapsed=348.8s
[collect] ep=500/1000 transitions=89354 success_rate=78.40% elapsed=348.8s
[collect] ep=625/1000 transitions=110984 success_rate=77.76% elapsed=348.8s
[collect] ep=750/1000 transitions=133057 success_rate=77.20% elapsed=348.8s
[collect] ep=875/1000 transitions=155243 success_rate=77.03% elapsed=348.8s
[collect] ep=1000/1000 transitions=177377 success_rate=76.80% elapsed=348.8s

[done] Saved 177377 transitions to offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz
  policy=goalseek  obs_dim=40  action_dim=2
  episodes=1000  success_rate=76.80%
  mean_return=-78.44 +/- 28.17

[collect] A3 → offline_data/privileged_s0_h4_efficiency_v2_re150_u10cr

### 2.3 A2 mix5050 — episode-level concat (spec §4.3)

按 spec 强制要求：goalseek 500 ep (seed=0) + crosscomp 500 ep (seed=1) 分别采集，再用 `scripts.concat_offline_datasets` 拼接，metadata 写入 `mix_components` / `mix_strategy="episode_level"` / `task_sampler="anchor_distribution"`。**不**用 `--policy-mixture`（spec deviation）。

In [ ]:
cfg = REGISTRY["A2"]
final_dir = Path("offline_data") / cfg.dataset_name

if (final_dir / "transitions.npz").exists():
    print(f"[skip] A2 mix already exists at {final_dir}")
else:
    sub_a = Path("offline_data") / "mix_tmp_goalseek_500ep_seed0"
    sub_b = Path("offline_data") / "mix_tmp_crosscomp_500ep_seed1"

    if not (sub_a / "transitions.npz").exists():
        print(f"\n[collect] A2 sub_a goalseek 500ep seed=0 → {sub_a}")
        !python -m scripts.collect_offline_data \
            --policy goalseek --seed 0 \
            --flow {cfg.flow_path} \
            --probe-layout {cfg.probe_layout} \
            --task-geometry {cfg.task_geometry} \
            --target-speed {cfg.target_speed} \
            --history-length 4 \
            --objective efficiency_v2 \
            --episodes 500 \
            --num-workers 8 \
            --output-dir {sub_a}

    if not (sub_b / "transitions.npz").exists():
        print(f"\n[collect] A2 sub_b crosscomp 500ep seed=1 → {sub_b}")
        !python -m scripts.collect_offline_data \
            --policy crosscomp --seed 1 \
            --flow {cfg.flow_path} \
            --probe-layout {cfg.probe_layout} \
            --task-geometry {cfg.task_geometry} \
            --target-speed {cfg.target_speed} \
            --history-length 4 \
            --objective efficiency_v2 \
            --episodes 500 \
            --num-workers 8 \
            --output-dir {sub_b}

    print(f"\n[concat] A2 mix5050 → {final_dir}")
    !python -m scripts.concat_offline_datasets \
        --input-dir {sub_a} \
        --input-dir {sub_b} \
        --output-dir {final_dir} \
        --mix-strategy episode_level \
        --task-sampler anchor_distribution



[collect] A2 sub_a goalseek 500ep seed=0 → offline_data/mix_tmp_goalseek_500ep_seed0
[collect] ep=63/500 transitions=10890 success_rate=74.60% elapsed=139.1s
[collect] ep=126/500 transitions=22435 success_rate=77.78% elapsed=154.5s
[collect] ep=189/500 transitions=34732 success_rate=82.01% elapsed=157.9s
[collect] ep=252/500 transitions=46444 success_rate=82.14% elapsed=157.9s
[collect] ep=315/500 transitions=57562 success_rate=80.63% elapsed=171.3s
[collect] ep=378/500 transitions=68172 success_rate=79.89% elapsed=171.3s
[collect] ep=441/500 transitions=78661 success_rate=78.00% elapsed=171.3s
[collect] ep=500/500 transitions=89354 success_rate=78.40% elapsed=171.3s

[done] Saved 89354 transitions to offline_data/mix_tmp_goalseek_500ep_seed0/transitions.npz
  policy=goalseek  obs_dim=40  action_dim=2
  episodes=500  success_rate=78.40%
  mean_return=-77.41 +/- 27.11

[collect] A2 sub_b crosscomp 500ep seed=1 → offline_data/mix_tmp_crosscomp_500ep_seed1
[collect] ep=63/500 transitions

### 2.4 写 sanity cards (spec §4.4)

7 个 unique datasets 各写一份 `sanity_card.json`（A2 与 A2-td3bc 共用同一 dataset，所以是 7 而非 8）。每张卡 7 字段：collector_success_rate / collector_mean_return / episode_length_mean+std / obs_dim / n_transitions / privileged_obs_present / flow_file。

In [ ]:
UNIQUE_DATASETS = {
    cfg.dataset_name: cfg.probe_layout
    for cfg in REGISTRY.values()
}
print(f"Writing sanity cards for {len(UNIQUE_DATASETS)} unique datasets...")
for dataset_name, probe in UNIQUE_DATASETS.items():
    dataset_dir = Path("offline_data") / dataset_name
    if not (dataset_dir / "transitions.npz").exists():
        print(f"[warn] missing dataset: {dataset_dir}")
        continue
    print(f"\n[sanity_card] {dataset_name}  (expected probe={probe})")
    !python -m scripts.write_sanity_card \
        --dataset-dir {dataset_dir} \
        --expected-probe-layout {probe}


Writing sanity cards for 7 unique datasets...

[sanity_card] goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000  (expected probe=s0)
[write] sanity card: offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/sanity_card.json
  collector_success_rate: 0.768
  collector_mean_return: -78.44115316126536
  episode_length_mean: 177.377
  episode_length_std: 60.80643774305481
  obs_dim: 40
  n_transitions: 177377
  privileged_obs_present: True
  flow_file: wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy
  n_episodes: 1000
  expected_probe_layout: s0
  history_length: 4
  expected_obs_dim: 40
  obs_dim_matches_probe_layout: True

[sanity_card] mix5050_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000  (expected probe=s0)
[write] sanity card: offline_data/mix5050_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/sanity_card.json
  collector_success_rate: 0.825
  collector_mean_return: -56.88471227961108
  episode_length_mean: 165.403
  episode_length_std: 53.876

### 2.5 自动 verdict — 任务 S1

In [4]:
import json

import pandas as pd

In [ ]:
import json

import pandas as pd

S1_SPOKE_ORDER = ["A1", "A2", "A2-td3bc", "A3", "B1", "B2", "C1", "C3"]

s1_rows = []
for spoke_id in S1_SPOKE_ORDER:
    cfg = REGISTRY[spoke_id]
    dataset_dir = Path("offline_data") / cfg.dataset_name
    card_path = dataset_dir / "sanity_card.json"
    if not card_path.exists():
        s1_rows.append({
            "spoke": spoke_id,
            "dataset": cfg.dataset_name,
            "success": None,
            "mean_R": None,
            "obs_dim": None,
            "obs_dim_match": None,
        })
        continue
    card = json.loads(card_path.read_text(encoding="utf-8"))
    s1_rows.append({
        "spoke": spoke_id,
        "dataset": cfg.dataset_name,
        "success": card.get("collector_success_rate"),
        "mean_R": card.get("collector_mean_return"),
        "obs_dim": card.get("obs_dim"),
        "obs_dim_match": card.get("obs_dim_matches_probe_layout"),
    })

s1_df = pd.DataFrame(s1_rows)
print("[per-spoke sanity card 摘要]")
print(s1_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print()

print("=" * 80)
print("任务 S1 verdict")
print("-" * 80)
all_present = all(r.get("obs_dim") is not None for r in s1_rows)
all_match   = all(r.get("obs_dim_match") is True for r in s1_rows)
print(f"  [cond 1] 8 spokes 三件套齐全 (transitions/metadata/sanity_card) : {'PASS' if all_present else 'FAIL'}")
print(f"  [cond 2] obs_dim 与 probe_layout 全部匹配 (s0=10/s1=12/s2=16)  : {'PASS' if all_match else 'FAIL'}")
print("=" * 80)


[per-spoke sanity card 摘要]
   spoke                                                           dataset  success    mean_R  obs_dim  obs_dim_match
      A1        goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000   0.7680  -78.4412       40           True
      A2         mix5050_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000   0.8250  -56.8847       40           True
A2-td3bc         mix5050_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000   0.8250  -56.8847       40           True
      A3      privileged_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000   0.9850   49.5124       40           True
      B1       crosscomp_s1_h4_efficiency_v2_re150_u10cross_fixdone_ep1000   0.8700  -36.4795       48           True
      B2       crosscomp_s2_h4_efficiency_v2_re150_u10cross_fixdone_ep1000   0.8700  -36.4795       64           True
      C1    crosscomp_s0_h4_efficiency_v2_re150_u10upstream_fixdone_ep1000   1.0000 -108.2119       40           True
      C3 crosscomp_s0_h4_effi

## 3. 任务 S2 — P1 训练 (16 runs, ~8h)

### 3.1 scope / judgement

8 spokes × 2 seeds (42 + 44 forced；seed 44 是 mainline 已知 outlier seed，不能 cherry-pick 排除)：

| 对照 | spoke 数 | algo | hyperparams | seeds | 备注 |
|---|---:|---|---|---|---|
| ReBRAC | 7 (A1/A2/A3/B1/B2/C1/C3) | rebrac | β1=4, β2=2 (anchor finalist) | 42, 44 | 单 cell |
| TD3+BC sanity (A2 only) | 1 (A2-td3bc) | td3bc | α=0.25 (phase0c Stage C 1000ep winner) | 42, 44 | 与 A2 ReBRAC 共用 mix5050 dataset |

预算：~8h L4 (16 runs × ~30 min each)。

锚定：anchor (Stage C finalist) **mean=0.902, std=0.021**（5 seeds × test=100；spec §3.1）。

触发判定 (spec §6.1)：

| 条件 | 阈值 | 解读 |
|---|---|---|
| `\|Δmean(spoke 2-seed) − 0.902\| > 0.05` | ±5pp | mean shift（双向，正负都触发） |
| `std(spoke 2-seed) > 2 × 0.021 = 0.042` | 4.2pp | std blow-up |
| `\|mean_ReBRAC_A2 − mean_TD3BC_A2\| < 0.05` | < 5pp | A2 only：gap collapse（ReBRAC vs TD3+BC 在 mix dataset 上无显著差距） |

### 3.2 P1 训练（8 spokes × 2 seeds 串行，driver 自带 skip-resume）

In [ ]:
P1_ORDER = ["A1", "A2", "A2-td3bc", "A3", "B1", "B2", "C1", "C3"]
for spoke_id in P1_ORDER:
    os.environ["SPOKE_ID"] = spoke_id
    os.environ["PHASE"]    = "p1"
    print(f"\n========== P1: {spoke_id} ==========")
    !bash scripts/run_offline_rebrac_broad.sh



========== P1: A1 ==========
[skip] dataset exists: offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz

[cmd] python3 -m scripts.generate_standard_benchmarks --benchmarks single_u10_cross_tgt15 --episodes 40 --output-dir benchmarks/offline_rebrac_broad/val_40
[done] single_u10_cross_tgt15 -> benchmarks/offline_rebrac_broad/val_40/single_u10_cross_tgt15.json

[cmd] python3 -m scripts.generate_standard_benchmarks --benchmarks single_u10_cross_tgt15 --episodes 100 --output-dir benchmarks/offline_rebrac_broad/test_100
[done] single_u10_cross_tgt15 -> benchmarks/offline_rebrac_broad/test_100/single_u10_cross_tgt15.json

[cmd] python3 -m scripts.train_offline --algo rebrac --offline-data offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz --flow wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy --manifest benchmarks/offline_rebrac_broad/val_40/single_u10_cross_tgt15.json --probe-layout s0 --history-lengt

### 3.3 P1 summarize + trigger gate (spec §6.1)

In [ ]:
!python -m scripts.summarize_broad_validation


[summary] P1 spokes summarized: 8
[summary] triggered for P2: 5
  - A1: mean=0.785 std=0.035 delta_pp=-0.117 reasons=mean_shift
  - A2: mean=0.730 std=0.040 delta_pp=-0.172 reasons=mean_shift
  - A3: mean=0.755 std=0.025 delta_pp=-0.147 reasons=mean_shift
  - C1: mean=0.225 std=0.005 delta_pp=-0.677 reasons=mean_shift
  - C3: mean=0.700 std=0.020 delta_pp=-0.202 reasons=mean_shift


In [5]:
SUMMARIES_DIR = Path("results/offline/rebrac/broad_validation/summaries")

In [ ]:
SUMMARIES_DIR = Path("results/offline/rebrac/broad_validation/summaries")

p1_summary = json.loads((SUMMARIES_DIR / "p1_overview.json").read_text(encoding="utf-8"))
p1_df = pd.DataFrame(p1_summary)
print("[P1 overview — per (spoke, pair)]")
print(p1_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))


[P1 overview — per (spoke, pair)]
spoke_id                    pair  num_seeds    seeds  mean_test_success_rate  std_test_success_rate  mean_test_return  std_test_return
      A1 actorb_4p0__criticb_2p0          2 [42, 44]                  0.7850                 0.0350         -105.0585          21.6421
      A2 actorb_4p0__criticb_2p0          2 [42, 44]                  0.7300                 0.0400          -69.6581           9.7333
A2-td3bc              alpha_0p25          2 [42, 44]                  0.6600                 0.0100          -95.9745           4.5609
      A3 actorb_4p0__criticb_2p0          2 [42, 44]                  0.7550                 0.0250           11.2000           3.0134
      B1 actorb_4p0__criticb_2p0          2 [42, 44]                  0.9300                 0.0100          -40.9564           1.2400
      B2 actorb_4p0__criticb_2p0          2 [42, 44]                  0.9100                 0.0300          -38.2518           7.1795
      C1 actorb_4p0__

### 3.4 自动 verdict — 任务 S2

In [6]:
ANCHOR_MEAN, ANCHOR_STD = 0.902, 0.021

In [ ]:
ANCHOR_MEAN, ANCHOR_STD = 0.902, 0.021

decisions = json.loads((SUMMARIES_DIR / "trigger_decisions.json").read_text(encoding="utf-8"))
triggered   = [d for d in decisions if d["triggered"]]
untriggered = [d for d in decisions if not d["triggered"]]

print("=" * 88)
print(f"P1 trigger gate (anchor mean={ANCHOR_MEAN:.3f}, std={ANCHOR_STD:.3f})")
print("-" * 88)
print(f"\nTriggered spokes ({len(triggered)}/{len(decisions)}):")
for d in triggered:
    print(f"  {d['spoke_id']:<10} mean={d['mean_test_success_rate']:.4f}  "
          f"std={d['std_test_success_rate']:.4f}  "
          f"Δ={d['delta_pp']*100:+.2f}pp  reasons={','.join(d['reasons'])}")

print(f"\nUntriggered spokes ({len(untriggered)}/{len(decisions)}):")
for d in untriggered:
    print(f"  {d['spoke_id']:<10} mean={d['mean_test_success_rate']:.4f}  "
          f"std={d['std_test_success_rate']:.4f}  "
          f"Δ={d['delta_pp']*100:+.2f}pp")
print("=" * 88)

if not triggered:
    print("\n→ 所有 spokes untriggered。整段 §4 (S3) 跳过；直接进 §6 报告写入清单。")
else:
    print(f"\n→ {len(triggered)} spokes 触发 P2；继续 §4 (S3)。")


P1 trigger gate (anchor mean=0.902, std=0.021)
----------------------------------------------------------------------------------------

Triggered spokes (5/7):
  A1         mean=0.7850  std=0.0350  Δ=-11.70pp  reasons=mean_shift
  A2         mean=0.7300  std=0.0400  Δ=-17.20pp  reasons=mean_shift
  A3         mean=0.7550  std=0.0250  Δ=-14.70pp  reasons=mean_shift
  C1         mean=0.2250  std=0.0050  Δ=-67.70pp  reasons=mean_shift
  C3         mean=0.7000  std=0.0200  Δ=-20.20pp  reasons=mean_shift

Untriggered spokes (2/7):
  B1         mean=0.9300  std=0.0100  Δ=+2.80pp
  B2         mean=0.9100  std=0.0300  Δ=+0.80pp

→ 5 spokes 触发 P2；继续 §4 (S3)。


## 4. 任务 S3 — P2 conditional deepening (~5–8h)

### 4.1 scope / judgement

只对 §3.4 触发的 spokes 跑：

| 步骤 | 内容 | seeds | 预算 |
|---|---|---|---|
| **S3.a β refit** | 每 spoke 跑 2 个相邻 cell `(β1=2, β2=2)` 与 `(β1=4, β2=1)` | 42 only | 1 spoke ≈ 2 runs ≈ 1h |
| **S3.b winner select** | 看 P1 + refit 共 3 个 cell 的 1-seed `seed=42` 数字，挑 winner | — | 几分钟手工 |
| **S3.c 5-seed 扩展** | winner config 跑 seeds 43/45/46（42 已有，44 大概率已有）| 42, 43, 44, 45, 46 (skip 已有) | 1 spoke ≈ 3 runs ≈ 1.5h |

**Winner drift 保护** (spec §6.2)：
- 若 best refit cell mean 与 anchor cell mean 差距 `< WINNER_DRIFT_THRESHOLD = 0.03` (3pp)，**保留 anchor `(β1=4, β2=2)`** 作为 winner（避免单 seed 噪声驱动 cell 切换）
- 5-seed 扩展前必须先 refit；**不允许**直接对原 anchor cell 做 5-seed（否则 P1 vs P2 不可分辨）

预算：1 spoke 全套 ≈ 2.5h；最多 3-4 spokes 触发，~5–8h。

### 4.2 S3.a — β refit (1-seed)

**手动编辑** `TRIGGERED_SPOKES`：从 §3.4 cell 输出复制 spoke ID。

In [ ]:
# EDIT THIS LIST after running §3.4.
# 例如：TRIGGERED_SPOKES = ["A1", "C3"]
TRIGGERED_SPOKES: list[str] = ["A1", "A2", "A3", "C1", "C3"]

REFIT_GRID = [("2.0", "2.0"), ("4.0", "1.0")]

for spoke_id in TRIGGERED_SPOKES:
    for actor_b, critic_b in REFIT_GRID:
        os.environ["SPOKE_ID"]             = spoke_id
        os.environ["PHASE"]                = "p2_refit"
        os.environ["ACTOR_PENALTY_COEFS"]  = actor_b
        os.environ["CRITIC_PENALTY_COEFS"] = critic_b
        print(f"\n========== P2 refit: {spoke_id} (β1={actor_b}, β2={critic_b}) ==========")
        !bash scripts/run_offline_rebrac_broad.sh



========== P2 refit: A1 (β1=2.0, β2=2.0) ==========
[skip] dataset exists: offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz
[skip] manifest exists: benchmarks/offline_rebrac_broad/val_40/single_u10_cross_tgt15.json
[skip] manifest exists: benchmarks/offline_rebrac_broad/test_100/single_u10_cross_tgt15.json
[skip] trained run exists: checkpoints/offline/rebrac/broad_validation/A1/actorb_2p0__criticb_2p0/seed_42
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_2p0__criticb_2p0/validation/seed_42/agent_step_00005544.json
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_2p0__criticb_2p0/validation/seed_42/agent_step_00011088.json
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_2p0__criticb_2p0/validation/seed_42/agent_step_00016632.json
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_2p0__criticb_2p0/validation/seed_42/agent_step_00022176.jso

### 4.3 S3.b — winner select (查看 1-seed refit 数字)

跑完 §4.2 后，每个 triggered spoke 看 3 个 cell（anchor `(4,2)` from P1 + refit `(2,2)` + refit `(4,1)`）的 `seed=42` 1-seed test 数字，挑 mean 最高 + 离 anchor 0.902 最近为 winner。

`WINNER_DRIFT_THRESHOLD = 0.03`：若 best refit cell mean − anchor cell mean `< 3pp`，**保留 anchor (4.0, 2.0)** 作为 winner（避免单 seed 噪声驱动）。

In [ ]:
WINNER_DRIFT_THRESHOLD = 0.03

CANDIDATE_PAIRS = [
    ("anchor",  "actorb_4p0__criticb_2p0"),
    ("refit_a", "actorb_2p0__criticb_2p0"),
    ("refit_b", "actorb_4p0__criticb_1p0"),
]

print("=" * 88)
print(f"{'spoke':<10}{'tag':<10}{'pair':<32}{'seed':>6}{'test_succ':>11}{'Δ vs anchor':>14}")
print("-" * 88)

for spoke_id in TRIGGERED_SPOKES:
    rows = []
    for tag, pair in CANDIDATE_PAIRS:
        test_path = (Path("results/offline/rebrac/broad_validation") /
                     spoke_id / pair / "test" / "seed_42.json")
        if not test_path.exists():
            print(f"{spoke_id:<10}{tag:<10}{pair:<32}{'42':>6}{'MISSING':>11}{'':>14}")
            continue
        d = json.loads(test_path.read_text(encoding="utf-8"))
        succ = d["eval_success_rate"]
        rows.append((tag, pair, succ))
        print(f"{spoke_id:<10}{tag:<10}{pair:<32}{'42':>6}{succ:>11.4f}{(succ - ANCHOR_MEAN)*100:>+12.2f}pp")
    if rows:
        anchor_succ = next((s for tag, _, s in rows if tag == "anchor"), None)
        best_tag, best_pair, best_succ = max(rows, key=lambda r: r[2])
        if anchor_succ is not None and (best_succ - anchor_succ) < WINNER_DRIFT_THRESHOLD:
            print(f"  → drift={best_succ - anchor_succ:+.4f} < {WINNER_DRIFT_THRESHOLD:.2f} → 保留 anchor (4.0, 2.0)")
        else:
            print(f"  → winner = {best_tag} ({best_pair})")
    print()
print("=" * 88)


spoke     tag       pair                              seed  test_succ   Δ vs anchor
----------------------------------------------------------------------------------------
A1        anchor    actorb_4p0__criticb_2p0             42     0.7500      -15.20pp
A1        refit_a   actorb_2p0__criticb_2p0             42     0.7100      -19.20pp
A1        refit_b   actorb_4p0__criticb_1p0             42     0.8400       -6.20pp
  → winner = refit_b (actorb_4p0__criticb_1p0)

A2        anchor    actorb_4p0__criticb_2p0             42     0.6900      -21.20pp
A2        refit_a   actorb_2p0__criticb_2p0             42     0.4500      -45.20pp
A2        refit_b   actorb_4p0__criticb_1p0             42     0.6600      -24.20pp
  → drift=+0.0000 < 0.03 → 保留 anchor (4.0, 2.0)

A3        anchor    actorb_4p0__criticb_2p0             42     0.7800      -12.20pp
A3        refit_a   actorb_2p0__criticb_2p0             42     0.7500      -15.20pp
A3        refit_b   actorb_4p0__criticb_1p0             42

### 4.4 S3.c — 5-seed 扩展

**手动编辑** `WINNERS`：根据 §4.3 数字挑出每个 triggered spoke 的 winner cell。Driver skip-resume 会跳过已有的 seed (42/44 in P1)。

In [ ]:
# EDIT THIS DICT after inspecting §4.3 output.
# Format: {"<spoke_id>": ("<β1>", "<β2>"), ...}
# 例如：WINNERS = {"A1": ("4.0", "2.0"), "C3": ("2.0", "2.0")}
WINNERS: dict[str, tuple[str, str]] = {"A1": ("4.0", "1.0"),
                                       "A2": ("4.0", "2.0"),
                                       "A3": ("4.0", "1.0"),
                                       "C1": ("4.0", "2.0"),
                                       "C3": ("4.0", "2.0")
                                       }

for spoke_id, (actor_b, critic_b) in WINNERS.items():
    os.environ["SPOKE_ID"]             = spoke_id
    os.environ["PHASE"]                = "p2_5seed"
    os.environ["ACTOR_PENALTY_COEFS"]  = actor_b
    os.environ["CRITIC_PENALTY_COEFS"] = critic_b
    print(f"\n========== P2 5-seed: {spoke_id} (β1={actor_b}, β2={critic_b}) ==========")
    !bash scripts/run_offline_rebrac_broad.sh



========== P2 5-seed: A1 (β1=4.0, β2=1.0) ==========
[skip] dataset exists: offline_data/goalseek_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz
[skip] manifest exists: benchmarks/offline_rebrac_broad/val_40/single_u10_cross_tgt15.json
[skip] manifest exists: benchmarks/offline_rebrac_broad/test_100/single_u10_cross_tgt15.json
[skip] trained run exists: checkpoints/offline/rebrac/broad_validation/A1/actorb_4p0__criticb_1p0/seed_42
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_4p0__criticb_1p0/validation/seed_42/agent_step_00005544.json
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_4p0__criticb_1p0/validation/seed_42/agent_step_00011088.json
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_4p0__criticb_1p0/validation/seed_42/agent_step_00016632.json
[skip] validation exists: results/offline/rebrac/broad_validation/A1/actorb_4p0__criticb_1p0/validation/seed_42/agent_step_00022176.js

### 4.5 P2 final summarize

In [ ]:
!python -m scripts.summarize_broad_validation


[summary] P1 spokes summarized: 18
[summary] triggered for P2: 5
  - A1: mean=0.794 std=0.029 delta_pp=-0.108 reasons=mean_shift
  - A2: mean=0.606 std=0.157 delta_pp=-0.296 reasons=mean_shift,std_blow_up
  - A3: mean=0.734 std=0.053 delta_pp=-0.168 reasons=mean_shift,std_blow_up
  - C1: mean=0.218 std=0.007 delta_pp=-0.684 reasons=mean_shift
  - C3: mean=0.576 std=0.148 delta_pp=-0.326 reasons=mean_shift,std_blow_up


In [7]:
# 总览：每个 (spoke, pair) 的 n-seed mean / std / Δ vs anchor
p2_summary = json.loads((SUMMARIES_DIR / "p1_overview.json").read_text(encoding="utf-8"))

print("=" * 96)
print(f"{'spoke':<10}{'pair':<32}{'n_seeds':>9}{'mean':>9}{'std':>9}{'Δ vs anchor':>14}{'flag':>10}")
print("-" * 96)

for row in p2_summary:
    spoke = row["spoke_id"]
    pair  = row["pair"]
    n     = row["num_seeds"]
    m     = row["mean_test_success_rate"] or 0.0
    s     = row["std_test_success_rate"] or 0.0
    d     = m - ANCHOR_MEAN
    flag  = "5-seed" if n >= 5 else ""
    print(f"{spoke:<10}{pair:<32}{n:>9}{m:>9.4f}{s:>9.4f}{d*100:>+12.2f}pp{flag:>10}")
print("=" * 96)


spoke     pair                              n_seeds     mean      std   Δ vs anchor      flag
------------------------------------------------------------------------------------------------
A1        actorb_2p0__criticb_2p0                 1   0.7100   0.0000      -19.20pp          
A1        actorb_4p0__criticb_1p0                 5   0.7940   0.0287      -10.80pp    5-seed
A1        actorb_4p0__criticb_2p0                 2   0.7850   0.0350      -11.70pp          
A2        actorb_2p0__criticb_2p0                 1   0.4500   0.0000      -45.20pp          
A2        actorb_4p0__criticb_1p0                 1   0.6600   0.0000      -24.20pp          
A2        actorb_4p0__criticb_2p0                 5   0.6060   0.1573      -29.60pp    5-seed
A2-td3bc  alpha_0p25                              2   0.6600   0.0100      -24.20pp          
A3        actorb_2p0__criticb_2p0                 1   0.7500   0.0000      -15.20pp          
A3        actorb_4p0__criticb_1p0                 5   0.7

## 5. 自动 verdict — 整个 broad validation

### 5.1 主结果汇总（八个 spoke 各自 representative cell + per-axis takeaway）

每个 spoke 取 `n_seeds` 最大的 cell 作为 representative（5-seed > 2-seed），生成 axis-level 总览。

In [8]:
spoke_best = {}
for row in p2_summary:
    sid = row["spoke_id"]
    cur = spoke_best.get(sid)
    if cur is None or row["num_seeds"] > cur["num_seeds"]:
        spoke_best[sid] = row

axis_map = {
    "A1": "A", "A2": "A", "A2-td3bc": "A", "A3": "A",
    "B1": "B", "B2": "B",
    "C1": "C", "C3": "C",
}

print("=" * 96)
print(f"{'axis':<6}{'spoke':<10}{'pair':<32}{'n':>4}{'mean':>9}{'std':>9}{'Δ vs anchor':>14}")
print("-" * 96)

for sid in ["A1", "A2", "A2-td3bc", "A3", "B1", "B2", "C1", "C3"]:
    row = spoke_best.get(sid)
    if row is None:
        print(f"{axis_map[sid]:<6}{sid:<10}{'MISSING':<32}")
        continue
    m = row["mean_test_success_rate"] or 0.0
    s = row["std_test_success_rate"]  or 0.0
    print(f"{axis_map[sid]:<6}{sid:<10}{row['pair']:<32}{row['num_seeds']:>4}"
          f"{m:>9.4f}{s:>9.4f}{(m - ANCHOR_MEAN)*100:>+12.2f}pp")
print("=" * 96)
print(f"\nAnchor (Stage C, crosscomp/s0/cross_stream): {ANCHOR_MEAN:.3f} ± {ANCHOR_STD:.3f} (5 seeds)")


axis  spoke     pair                               n     mean      std   Δ vs anchor
------------------------------------------------------------------------------------------------
A     A1        actorb_4p0__criticb_1p0            5   0.7940   0.0287      -10.80pp
A     A2        actorb_4p0__criticb_2p0            5   0.6060   0.1573      -29.60pp
A     A2-td3bc  alpha_0p25                         2   0.6600   0.0100      -24.20pp
A     A3        actorb_4p0__criticb_1p0            5   0.7340   0.0531      -16.80pp
B     B1        actorb_4p0__criticb_2p0            2   0.9300   0.0100       +2.80pp
B     B2        actorb_4p0__criticb_2p0            2   0.9100   0.0300       +0.80pp
C     C1        actorb_4p0__criticb_2p0            5   0.2180   0.0075      -68.40pp
C     C3        actorb_4p0__criticb_2p0            5   0.5760   0.1479      -32.60pp

Anchor (Stage C, crosscomp/s0/cross_stream): 0.902 ± 0.021 (5 seeds)


## 6. 报告写入清单

跑完上面所有 cell 后，按下表更新文档：

| 任务 | 写入位置 | 内容 |
|---|---|---|
| **S1** | `docs/rebrac_broad_validation_report.md §2`（NEW） | 7 个 dataset 的 sanity card 数字（succ / mean_R / obs_dim / mix_components） |
| **S2** | `docs/rebrac_broad_validation_report.md §3` | 8 spoke × 2 seed P1 数字表 + trigger 决定（哪些进 P2、原因） |
| **S3** | `docs/rebrac_broad_validation_report.md §4` | 触发 spoke 的 β refit 1-seed × 3 cell + 5-seed final 表（含 winner 标注） |
| **总览** | `docs/rebrac_broad_validation_report.md §5` | per-axis takeaway（A 三 spokes：data quality 影响；B 两 spokes：sensor envelope；C 两 spokes：task geometry） |
| **§6 limitations** | `docs/rebrac_broad_validation_report.md §6` | 5-seed underpowered；C2 Re250 cut 原因（spec §3.5）；A2 mix5050 等比 vs 真实部署分布的差距 |
| **§0 abstract** | `docs/rebrac_broad_validation_report.md §0`（最后写）| 一段 4–5 句蒸馏 §3–§5 |
| **cross-link** | `docs/rebrac_mainline_review.md §3.5` | append: "broad validation 已完成，详见 docs/rebrac_broad_validation_report.md" + spec/plan/report 三个 link |
| **paper §x.y** | paper main text robustness subsection | 视触发情况：若 ≥1 spoke 触发，paper discussion 加一段 "robustness 验证表明 ... 在 ... 维度上结果保持/退化"；若全部 untriggered，一句话 "广验在所有 8 spokes 上保持 anchor 数字" |

完成后执行：
```bash
git add docs/rebrac_broad_validation_report.md docs/rebrac_mainline_review.md \
        results/offline/rebrac/broad_validation/summaries/p1_overview.csv \
        results/offline/rebrac/broad_validation/summaries/p1_overview.json \
        results/offline/rebrac/broad_validation/summaries/trigger_decisions.json \
        notebooks/rebrac_broad_validation.ipynb
git commit -m "docs(rebrac-broad): broad-validation 报告完成（spec §3 + §4 + §5）"
```

之后即可在 paper main results 之外加一段 robustness subsection（spec §11），不必再回到 broad validation。
